# 4 - Analyses statistiques (proportion)

L'objectif ici est de reproduire toutes les analyses statistiques de base sur la "République" en utilisant les occurrences en proportion de l'unité de mesure totale. 
Pour cela on prend le fichier de base auquel on a ajouté la colonne avec le résultat de la regex en true/false. 

*À garder en tête au moment de l'analyse et interprétation des résultats, l'analyse en % dépend de notre unité de mesure (nombre de prises de paroles avec ou sans interruption, phrases). On ne peut pas mesurer en durée de l'intervention ou nombre de mots pour l'intervention donc il ne s'agit pas à proprement parlé d'un % en termes de temps de parole (= un artéfact statistique dont il serait intéressant de comparer les mesures).*

==> ***Croiser analyses à l'échelle des prises de paroles regroupées ou non, et à l'échelle des phrases (en utilisant Spacy)***

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv(
    "../data/interim/df_regroup_repu_proportion.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [3]:
import datetime
import locale

# Active la locale française (nécessaire pour le format)
locale.setlocale(locale.LC_TIME, "fr_FR.UTF-8")

'fr_FR.UTF-8'

In [4]:
df["dateSeance_ts"] = pd.to_datetime(df["dateSeanceJour"], format="%A %d %B %Y")
df["dateSeance_day"] = df["dateSeance_ts"].dt.normalize()  

## Analyses générales temporelles

### Par jours

#### Par jours sur corpus intégral

In [5]:
# Compter True/False par jour
df_daily = (
    df.groupby(df["dateSeance_day"].dt.date)["repu_match_valide"]
    .value_counts(normalize=True)  # calcule directement les proportions
    .rename("proportion")
    .reset_index()
)

# Garder uniquement les "True"
df_daily_true = df_daily[df_daily["repu_match_valide"] == True]

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_daily_true.sort_values("proportion", ascending=False).head(20)
table

,dateSeance_day,repu_match_valide,proportion
4,2017-07-03,True,0.470588
1908,2024-03-04,True,0.352941
1236,2021-06-28,True,0.278027
839,2020-03-22,True,0.266667
1084,2021-02-01,True,0.260204
316,2018-07-09,True,0.227273
1092,2021-02-05,True,0.217391
500,2019-02-11,True,0.174603
1088,2021-02-03,True,0.171990
1377,2022-01-06,True,0.168000


**Remarques**
*Le 3 juillet 2017, le 9 juillet 2018, le 4 mars 2024 sont parmis les 5 principales dates car parlement réuni en Congrès*

*le 1 février 2021, le 28 juin 2021, 5 février 2021, le 23 juillet 2021, le 3 février 2021, 30 juin 2021 (et 5 avril 2023 : bilan de la loi), 1er juillet 2021, le 12 février 2021, renvoient quant à eux à la discussion du projet de loi "confortant le respect des principes de la République". **==> Suivre en détail le processus législatif de ce projet de loi car moment central !!***
*On a aussi le 22 mars 2020, très courte séance (commencée à 18h30) sur l’urgence du covid et un hommage*

*6 janvier 2022, 6 juin 2022, 13 mars 2018 : réforme territoriale Nouvelle-Calédonie*

*25 janvier 2024, 8 juillet 2019 et 16 janvier 2020 : accords internationaux avec présence d'expressions comme "gouvernement de la République française", de pays sous forme adjectivable (ex : "république arménienne") ou avec république en miniscule --> moins présent maintenant que exclus*

*11 février 2019 sur "l'école de la confiance"*
*12 et 13 juillet 2018 sur le  projet de loi constitutionnelle pour une Démocratie plus représentative, responsable et efficace*

In [6]:
# Compter True/False par jour
df_daily = (
    df.groupby(df["dateSeance_day"].dt.date)["repu_match_valide"]
    .value_counts(normalize=True)  # calcule directement les proportions
    .rename("proportion")
    .reset_index()
)

# Garder uniquement les "True"
df_daily_true = df_daily[df_daily["repu_match_valide"] == True]

# Graphique
fig_daily = px.line(
    df_daily_true,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par jour",
    labels={"proportion": "% des occurences", "dateSeance_day": "Date"},
    template="plotly_white",
)

fig_daily.show()


#### Par jour sur corpus annualisé 

In [7]:
# Choisir une année
annee = 2022

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["dateSeance_day"].dt.year == annee]

# Compter True/False par jour
df_daily = (
    df_annee.groupby(df["dateSeance_day"].dt.date)["repu_match_valide"]
    .value_counts(normalize=True)  # calcule directement les proportions
    .rename("proportion")
    .reset_index()
)

# Garder uniquement les "True"
df_daily_true = df_daily[df_daily["repu_match_valide"] == True]

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_daily_true.sort_values("proportion", ascending=False).head(50)
table

,dateSeance_day,repu_match_valide,proportion
7,2022-01-06,True,0.168000
47,2022-02-09,True,0.117647
41,2022-02-03,True,0.113122
53,2022-02-16,True,0.108333
35,2022-01-31,True,0.102564
92,2022-07-27,True,0.086735
64,2022-03-01,True,0.079365
62,2022-02-24,True,0.072816
29,2022-01-25,True,0.069444
43,2022-02-04,True,0.068592


In [8]:
# Graphique
fig_daily = px.line(
    df_daily_true,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par jour",
    labels={"proportion": "% des occurences", "dateSeance_day": "Date"},
    template="plotly_white",
)

fig_daily.show()

### Par semaine

#### Par semaine sur corpus intégral

In [9]:
# Grouper par semaine
df_weekly = (
    df.groupby(df["dateSeance_day"].dt.to_period("W"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_weekly["dateSeance_day"] = df_weekly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_weekly_true = df_weekly[df_weekly["repu_match_valide"] == True]

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_weekly_true.sort_values("proportion", ascending=False).head(20)
table

,dateSeance_day,repu_match_valide,proportion
293,2021-02-01,True,0.166391
329,2021-06-28,True,0.164981
85,2018-07-09,True,0.126969
67,2018-05-07,True,0.112994
59,2018-03-26,True,0.103359
3,2017-07-03,True,0.096983
295,2021-02-08,True,0.096909
205,2020-01-13,True,0.094545
131,2019-02-11,True,0.084378
381,2022-02-28,True,0.079365


**Remarques**
- 2 premières semaines (+ S6) correspondent aux débats sur la loi « confortant le respect des principes de la République et la lutte contre le séparatisme » 
- 3e semaine correspond au débat sur la démocratie représentative

In [10]:
# Grouper par semaine
df_weekly = (
    df.groupby(df["dateSeance_day"].dt.to_period("W"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_weekly["dateSeance_day"] = df_weekly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_weekly_true = df_weekly[df_weekly["repu_match_valide"] == True]

# Graphique
fig_weekly = px.bar(
    df_weekly_true,
    x="dateSeance_day",
    y="proportion",
    title="Évolution de la proportion des occurrences de la famille du mot 'République' par semaine",
    labels={"proportion": "% des occurences hebdomaire", "dateSeance_day": "Semaine"},
    template="plotly_white",
)

fig_weekly.show()


#### Par semaine sur corpus annualisé

In [11]:
# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee = df[df["dateSeance_day"].dt.year == annee]

# Grouper par semaine
df_weekly = (
    df_annee.groupby(df["dateSeance_day"].dt.to_period("W"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_weekly["dateSeance_day"] = df_weekly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_weekly_true = df_weekly[df_weekly["repu_match_valide"] == True]

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_weekly_true.sort_values("proportion", ascending=False).head(20)
table

,dateSeance_day,repu_match_valide,proportion
7,2021-02-01,True,0.166391
43,2021-06-28,True,0.164981
9,2021-02-08,True,0.096909
51,2021-09-06,True,0.062992
69,2021-11-15,True,0.059680
3,2021-01-18,True,0.054820
15,2021-03-08,True,0.052557
35,2021-05-31,True,0.047913
19,2021-03-22,True,0.045631
47,2021-07-12,True,0.044843


In [12]:
# Grouper par semaine
df_weekly = (
    df.groupby(df["dateSeance_day"].dt.to_period("W"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_weekly["dateSeance_day"] = df_weekly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_weekly_true = df_weekly[df_weekly["repu_match_valide"] == True]

# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee_semaine = df_weekly_true[df_weekly_true["dateSeance_day"].dt.year == annee]

# Graphique
fig_annee_semaine = px.bar(
    df_annee_semaine,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par semaine en 2024",
    labels={"proportion": "% des occurences mensualisées", "dateSeance_day": "Mois"},
    template="plotly_white",
)

fig_annee_semaine.show()

### Par mois

#### Par mois sur corpus intégral

In [13]:
# Grouper par mois
df_monthly = (
    df.groupby(df["dateSeance_day"].dt.to_period("M"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_monthly["dateSeance_day"] = df_monthly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_monthly_true = df_monthly[df_monthly["repu_match_valide"] == True]

# aficher les 10 mois les plus fréquents sous forme de tableau
table = df_monthly_true.sort_values("proportion", ascending=False).head(10)
table

,dateSeance_day,repu_match_valide,proportion
85,2021-02-01,True,0.098328
109,2022-03-01,True,0.075758
41,2019-02-01,True,0.068935
27,2018-07-01,True,0.064927
93,2021-06-01,True,0.055933
107,2022-02-01,True,0.055247
29,2018-08-01,True,0.054264
7,2017-09-01,True,0.053606
19,2018-03-01,True,0.048549
142,2023-12-01,True,0.043991


In [14]:
# Grouper par mois
df_monthly = (
    df.groupby(df["dateSeance_day"].dt.to_period("M"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_monthly["dateSeance_day"] = df_monthly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_monthly_true = df_monthly[df_monthly["repu_match_valide"] == True]

# Graphique
fig_monthly = px.bar(
    df_monthly_true,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par mois",
    labels={"proportion": "% des occurences mensualisées", "dateSeance_day": "Mois"},
    template="plotly_white",
)

fig_monthly.show()


#### Par mois sur corpus annualisé

In [15]:
# Grouper par mois
df_monthly = (
    df.groupby(df["dateSeance_day"].dt.to_period("M"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_monthly["dateSeance_day"] = df_monthly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_monthly_true = df_monthly[df_monthly["repu_match_valide"] == True]

# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee_mois = df_monthly_true[df_monthly_true["dateSeance_day"].dt.year == annee]

# aficher les 25 dates les plus fréquentes sous forme de tableau
table = df_annee_mois.sort_values("proportion", ascending=False).head(20)
table

,dateSeance_day,repu_match_valide,proportion
85,2021-02-01,True,0.098328
93,2021-06-01,True,0.055933
87,2021-03-01,True,0.035191
91,2021-05-01,True,0.033400
95,2021-07-01,True,0.033107
97,2021-09-01,True,0.031363
83,2021-01-01,True,0.028571
103,2021-12-01,True,0.027990
101,2021-11-01,True,0.025794
99,2021-10-01,True,0.020162


In [16]:
# Grouper par mois
df_monthly = (
    df.groupby(df["dateSeance_day"].dt.to_period("M"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_monthly["dateSeance_day"] = df_monthly["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_monthly_true = df_monthly[df_monthly["repu_match_valide"] == True]

# Choisir une année
annee = 2021

# Filtrer le DataFrame sur l'année choisie
df_annee_mois = df_monthly_true[df_monthly_true["dateSeance_day"].dt.year == annee]

# Graphique
fig_annee_mois = px.bar(
    df_annee_mois,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par mois en 2021",
    labels={"proportion": "% des occurences mensualisées", "dateSeance_day": "Mois"},
    template="plotly_white",
)

fig_annee_mois.show()

#### Par an

In [ ]:
# Grouper par an
df_annee = (
    df.groupby(df["dateSeance_day"].dt.to_period("YE"))["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Transformer la période en datetime (pour l'axe X)
df_annee["dateSeance_day"] = df_annee["dateSeance_day"].dt.to_timestamp()

# Garder True uniquement
df_annee_true = df_annee[df_annee["repu_match_valide"] == True]

# aficher les 10 mois les plus fréquents sous forme de tableau
table = df_annee_true.sort_values("proportion", ascending=False).head(10)
table

,dateSeance_day,repu_match_valide,proportion
9,2021-01-01,True,0.039641
3,2018-01-01,True,0.029975
1,2017-01-01,True,0.027344
5,2019-01-01,True,0.026378
11,2022-01-01,True,0.023629
13,2023-01-01,True,0.022428
15,2024-01-01,True,0.022087
7,2020-01-01,True,0.021771


In [18]:
# Grouper par année
df_yearly = (
    df.groupby(df["dateSeance_day"].dt.year)["repu_match_valide"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

# Garder True uniquement
df_yearly_true = df_yearly[df_yearly["repu_match_valide"] == True]

# Graphique
fig_yearly = px.bar(
    df_yearly_true,
    x="dateSeance_day",
    y="proportion",
    title="Proportion occurrences de la 'République' par année",
    labels={"proportion": "% des occurences annualisées", "dateSeance_day": "Année"},
    template="plotly_white",
)

fig_yearly.show()


## Par groupes 

### Sur total 

In [19]:
# Compter le nombre de fois où chaque groupe parlementaire dit "République"
counts = (
    df.groupby("groupe_députés_affiliation")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 250]

# Trier par proportion décroissante et garder les 40 premiers
df_toppartis = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig_toppartis = px.bar(
    df_toppartis,
    x="groupe_députés_affiliation",
    y="proportion_true",
    title="Proportions des occurrences de la 'République' par groupe parlementaire (<250 occurrences)",
    labels={"groupe_députés_affiliation": "Groupe parlementaire", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_toppartis.update_layout(xaxis_tickangle=-45)

fig_toppartis.show()
df_toppartis

,groupe_députés_affiliation,total_mentions,true_mentions,proportion_true
0,UDI,12165,426,0.035018
1,LFI,51624,1715,0.033221
2,GDR,31821,1012,0.031803
3,LIOT,11336,356,0.031404
4,SOC-A,29757,895,0.030077
5,ECO,9759,262,0.026847
6,REN,73626,1696,0.023035
7,DEM,26398,603,0.022843
8,RN,19541,372,0.019037
9,LR,98740,1682,0.017035


In [20]:
# TODO : changer manuellement les couleurs des graphiques

def analyse_evolution_pourcentage(df,
                                  date_col="dateSeance_day",
                                  parti_col="groupe_députés_affiliation",
                                  match_col="repu_match_valide",
                                  min_true_mentions=500,
                                  top_n=5):

    # Calcul global des proportions par groupe
    counts = (
        df.groupby(parti_col)[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion globale
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # Filtrer les partis avec au moins min_true_mentions
    filtered = counts[counts["true_mentions"] >= min_true_mentions]

    # Sélectionner les top_n groupes selon la proportion
    top_partis = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[parti_col]
        .tolist()
    )

    # Filtrer les données pour ces groupes
    df_top = df[df[parti_col].isin(top_partis)].copy()

    # Calculer les stats annuelles : mentions totales et vraies
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="Y"), parti_col])[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion annuelle (% d'utilisation)
    df_grouped["proportion_true"] = (
        df_grouped["true_mentions"] / df_grouped["total_mentions"]
    )

    # Extraire l'année pour affichage
    df_grouped["Année"] = df_grouped[date_col].dt.year

    # Tracer l'évolution du % dans le temps
    fig = px.line(
        df_grouped,
        x="Année",
        y="proportion_true",
        color=parti_col,
        markers=True,
        title=f"Évolution annuelle du % d'utilisation de la 'République' des {top_n} principaux groupes parlementaires (>{min_true_mentions} occurrences)",
        labels={
            "proportion_true": "% d'utilisation",
            parti_col: "Groupe parlementaire"
        }
    )

    fig.update_layout(
        xaxis=dict(dtick=1),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Groupe",
        yaxis_tickformat=".0%",
    )

    fig.show()


In [21]:
analyse_evolution_pourcentage(df)

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3294/1209360484.py:35: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



### Par groupe

In [22]:
# Définir les paramètres
parti = "REN"
colonne_condition = "repu_match_valide"
periode = "YS"  # "M" = mois, "W" = semaine, penser à YS pour avoir début d'année 

# Filtrer les données pour l'orateur
df_parti = df[df["groupe_députés_affiliation"] == parti].copy()

# S'assurer que la colonne date est bien en datetime
df_parti["dateSeance_day"] = pd.to_datetime(df_parti["dateSeance_day"])

# Agréger les données par mois
df_monthly = (
    df_parti
    .resample(periode, on="dateSeance_day")[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

# Calculer la proportion
df_monthly["proportion_true"] = df_monthly["true_mentions"] / df_monthly["total_mentions"]

# Créer la figure avec Plotly Graph Objects
fig = go.Figure()

# Barres : volume total
fig.add_trace(go.Bar(
    x=df_monthly["dateSeance_day"],
    y=df_monthly["true_mentions"],
    name="Nombre total des occurrences",
    marker_color="rgba(99, 110, 250, 0.6)",
    yaxis="y1"
))

# Courbe : proportion
fig.add_trace(go.Scatter(
    x=df_monthly["dateSeance_day"],
    y=df_monthly["proportion_true"] * 100,  # en pourcentage
    name="Proportions des occurrences",
    mode="lines+markers",
    line=dict(color="rgba(239, 85, 59, 0.8)", width=3),
    yaxis="y2"
))

# Mise en forme du graphique
fig.update_layout(
    title=f"Évolution mensuelle des occurrences de la 'République' par {parti}",
    xaxis=dict(title="Mois"),
    yaxis=dict(
        title="Occurrences de la 'République'",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion des occurences sur total",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.7)"),
    template="plotly_white",
    bargap=0.2
)

fig.show()


### Par période 

In [23]:
def top_parti_periodique(df, periode=None, annee=None, semaine=None, jour=None, 
                            date_debut=None, date_fin=None, jours=None,
                            colonne_condition="repu_match_valide",
                            seuil_min_true=None, top_n=20):

    # --- Filtrage selon la période ---
    if periode == "annee":
        if annee is None:
            raise ValueError("Il faut préciser l'année pour periode='annee'")
        df_filtered = df[df["dateSeance_day"].dt.year == annee]
        titre = f"Parti mobilisant en % le plus la 'République' en {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true} occurences)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["dateSeance_day"].dt.isocalendar().year == annee) &
            (df["dateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Parti mobilisant en % le plus la 'République' la {semaine} semaine {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true})"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df[df["dateSeance_day"].dt.date == jour_dt]
        titre = f"Parti mobilisant le plus en % la 'République' le {jour_dt} (Seuil de {seuil_min_true} occurences)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["dateSeance_day"] >= debut) & (df["dateSeance_day"] <= fin)]
        titre = f"Parti mobilisant le plus en % la 'République' du {debut.date()} au {fin.date()} (Seuil de {seuil_min_true} occurences)"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df["dateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Parti mobilisant le plus la 'République' en % lors de X évènement (Seuil de {seuil_min_true} occurences)"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des occurrences ---
    counts = (
        df_filtered.groupby("groupe_députés_affiliation")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # --- Filtrage par seuil ---
    filtered = counts[counts["true_mentions"] >= seuil_min_true]

    # --- Trier par proportion décroissante et garder top_n ---
    df_top = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # --- Graphique ---
    fig = px.bar(
        df_top,
        x="groupe_députés_affiliation",
        y="proportion_true",
        hover_data=["true_mentions", "total_mentions"],
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False, yaxis_title="Proportion des interventions totales", xaxis_title="Groupes parlementaires")
    fig.show()

    return df_top


In [24]:
# Top partis en % par année
top_parti_periodique(df, periode="annee", annee=2017, seuil_min_true=25)

,groupe_députés_affiliation,total_mentions,true_mentions,proportion_true
0,LFI,3986,202,0.050677
1,RN,618,25,0.040453
2,GDR,2677,101,0.037729
3,REN,4854,142,0.029254
4,UDI,2000,50,0.025000
5,DEM,1336,31,0.023204
6,SOC-A,2987,69,0.023100
7,LR,9850,150,0.015228


In [25]:
# Top 10 orateurs sur la semaine X de X
top_parti_periodique(df, periode="semaine", annee=2021, semaine=6, seuil_min_true=20)

,groupe_députés_affiliation,total_mentions,true_mentions,proportion_true
0,LFI,169,21,0.124260
1,REN,708,81,0.114407
2,DEM,296,25,0.084459
3,LR,881,48,0.054484


In [26]:
# Top  partis en % (tel jour) le X 
top_parti_periodique(df, periode="jour", jour="2021-02-01",seuil_min_true=10)

,groupe_députés_affiliation,total_mentions,true_mentions,proportion_true
0,REN,47,13,0.276596


In [27]:
# Top partis en % (sur telle période) du 1er au 16 février 2021
top_parti_periodique(df, periode="intervalle", date_debut="2021-02-01", date_fin="2021-02-16", seuil_min_true=20)

,groupe_députés_affiliation,total_mentions,true_mentions,proportion_true
0,AGIR-E,104,31,0.298077
1,UDI,201,34,0.169154
2,GDR,278,42,0.151079
3,SOC-A,223,33,0.147982
4,REN,1355,189,0.139483
5,DEM,520,63,0.121154
6,LIOT,198,21,0.106061
7,LFI,521,53,0.101727
8,LR,1659,110,0.066305


#### Évolutions par période

In [28]:
# TODO : changer manuellement les couleurs des graphiques

def groupes_evolutions_périodes(df,
                                  date_col="dateSeance_day",
                                  parti_col="groupe_députés_affiliation",
                                  match_col="repu_match_valide",
                                  min_true_mentions=100,
                                  top_n=7,
                                  start_year=2021,
                                  end_year=2021):

    # Filtrer par période si spécifiée
    if start_year or end_year:
        mask = pd.Series(True, index=df.index)
        if start_year:
            mask &= df[date_col].dt.year >= start_year
        if end_year:
            mask &= df[date_col].dt.year <= end_year
        df = df.loc[mask].copy()

    # Calcul global des proportions par groupe
    counts = (
        df.groupby(parti_col)[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # Filtrer les groupes pertinents
    filtered = counts[counts["true_mentions"] >= min_true_mentions]

    # Sélectionner les top groupes selon la proportion
    top_partis = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[parti_col]
        .tolist()
    )

    # Filtrer les données pour ces groupes
    df_top = df[df[parti_col].isin(top_partis)].copy()

    # Calcul annuel des proportions
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="M"), parti_col])[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    df_grouped["proportion_true"] = df_grouped["true_mentions"] / df_grouped["total_mentions"]
    df_grouped["Année"] = df_grouped[date_col].dt.year
    df_grouped["Mois"] = df_grouped[date_col].dt.strftime("%Y-%m")

    # Tracé du graphique (% d’utilisation par année)
    fig = px.line(
        df_grouped,
        x="Mois",
        y="proportion_true",
        color=parti_col,
        markers=True,
        title=(
            f"Évolution mensuelle du % d'utilisation du mot 'République' "
            f"(Des {top_n} principaux groupes parlementaires, {start_year or df_grouped['Année'].min()}–{end_year or df_grouped['Année'].max()})"
        ),
        labels={"proportion_true": "% d'utilisation", parti_col: "Groupe parlementaire"}
    )

    fig.update_layout(
        xaxis=dict(dtick="M1", tickangle=45),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Groupe",
        yaxis_tickformat=".0%"
    )

    fig.show()


In [29]:
groupes_evolutions_périodes(df)

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3294/1814616223.py:45: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



### Évolutions pour chaque groupe

## Par personnel politique / individuellement

### Les principaux orateurs sur la période 

In [30]:
# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df.groupby("nom_orateur_clean")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 65]

# Trier par proportion décroissante et garder les 40 premiers
df_top20 = filtered.sort_values("proportion_true", ascending=False).head(20).reset_index(drop=True)

# Graphique
fig_top20 = px.bar(
    df_top20,
    x="nom_orateur_clean",
    y="proportion_true",
    title="Top 20 orateurs par proportion de mentions de 'République'",
    labels={"nom_orateur_clean": "Personnel politique", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_top20.update_layout(xaxis_tickangle=-45)

fig_top20.show()
df_top20

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Édouard Philippe,396,95,0.239899
1,Mme Marlène Schiappa,517,98,0.189555
2,M. Jean-Michel Blanquer,766,142,0.185379
3,M. Éric Ciotti,989,120,0.121335
4,M. Sébastien Lecornu,890,101,0.113483
5,M. Gérald Darmanin,3154,325,0.103044
6,M. Jean-Christophe Lagarde,999,92,0.092092
7,Mme Élisabeth Borne,1102,74,0.067151
8,M. Dominique Potier,1819,119,0.065421
9,M. Alexis Corbière,3074,198,0.064411


### Évolutions

#### Évolution dans le temps de la fréquence d'utilisation du top X de personnels politiques

In [31]:
# TODO : changer manuellement les couleurs des graphiques

def evolution_personnel_pourcentage(df,
                                  date_col="dateSeance_day",
                                  personnel_col="nom_orateur_clean",
                                  match_col="repu_match_valide",
                                  min_true_mentions=150,
                                  top_n=6):

    # Calcul global des proportions par groupe
    counts = (
        df.groupby(personnel_col)[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion globale
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # Filtrer les partis avec au moins min_true_mentions
    filtered = counts[counts["true_mentions"] >= min_true_mentions]

    # Sélectionner les top_n groupes selon la proportion
    top_personnel = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[personnel_col]
        .tolist()
    )

    # Filtrer les données pour ces groupes
    df_top = df[df[personnel_col].isin(top_personnel)].copy()

    # Calculer les stats annuelles : mentions totales et vraies
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="Y"), personnel_col])[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion annuelle (% d'utilisation)
    df_grouped["proportion_true"] = (
        df_grouped["true_mentions"] / df_grouped["total_mentions"]
    )

    # Extraire l'année pour affichage
    df_grouped["Année"] = df_grouped[date_col].dt.year

    # Tracer l'évolution du % dans le temps
    fig = px.line(
        df_grouped,
        x="Année",
        y="proportion_true",
        color=personnel_col,
        markers=True,
        title=f"Évolution annuelle du % d'utilisation de la 'République' des {top_n} principaux personnels politiques (>{min_true_mentions} occurrences)",
        labels={
            "proportion_true": "% d'utilisation",
            personnel_col: "Groupe parlementaire"
        }
    )

    fig.update_layout(
        xaxis=dict(dtick=1),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Groupe",
        yaxis_tickformat=".0%",
    )

    fig.show()


In [32]:
evolution_personnel_pourcentage(df)

/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3294/434538138.py:35: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



#### Principaux orateurs par législatures

In [33]:
df_16e = df[df["legislature"]== 16]
df_15e = df[df["legislature"]== 15]

In [34]:
# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df_16e.groupby("nom_orateur_clean")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 40]

# Trier par proportion décroissante et garder les 40 premiers
df16_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig16_top20 = px.bar(
    df16_top20,
    x="nom_orateur_clean",
    y="proportion_true",
    title="Top 20 orateurs par proportion de mentions de 'République > 40, 16e législature'",
    labels={"nom_orateur_clean": "Personnel politique", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig16_top20.update_layout(xaxis_tickangle=-45)

fig16_top20.show()
df16_top20

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,Mme Élisabeth Borne,258,66,0.255814
1,M. Gérald Darmanin,686,115,0.167638
2,M. Antoine Léaument,1560,74,0.047436
3,M. Benjamin Lucas,2481,104,0.041919
4,M. Éric Dupond-Moretti,1557,48,0.030829


In [35]:
# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df_15e.groupby("nom_orateur_clean")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 100]

# Trier par proportion décroissante et garder les 40 premiers
df15_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig15_top20 = px.bar(
    df15_top20,
    x="nom_orateur_clean",
    y="proportion_true",
    title="Top 20 orateurs par proportion de mentions de 'République > 40 15e législature'",
    labels={"nom_orateur_clean": "Personnel politique", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig15_top20.update_layout(xaxis_tickangle=-45)

fig15_top20.show()
df15_top20

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Jean-Michel Blanquer,766,142,0.185379
1,M. Éric Ciotti,846,103,0.121749
2,M. Gérald Darmanin,2468,210,0.085089
3,M. Alexis Corbière,2365,178,0.075264
4,M. Jean-Luc Mélenchon,2601,157,0.060361
5,M. Stéphane Peu,2349,139,0.059174
6,M. Éric Coquerel,2774,141,0.050829
7,M. Sébastien Jumel,4007,183,0.045670
8,M. Philippe Gosselin,3019,111,0.036767
9,Mme Danièle Obono,3113,111,0.035657


### Les principaux orateurs par années/semaines/séances

In [36]:
def top_orateurs_periodique(df, periode="annee", annee=None, semaine=None, jour=None, 
                            date_debut=None, date_fin=None, jours=None,
                            colonne_condition="repu_match_valide",
                            seuil_min_true=None, top_n=20):

    # --- Filtrage selon la période ---
    if periode == "annee":
        if annee is None:
            raise ValueError("Il faut préciser l'année pour periode='annee'")
        df_filtered = df[df["dateSeance_day"].dt.year == annee]
        titre = f"Personnel politique mobilisant en % le plus la 'République' en {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true} occurences)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["dateSeance_day"].dt.isocalendar().year == annee) &
            (df["dateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Personnel politique mobilisant en % le plus la 'République' la {semaine} semaine {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true})"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df[df["dateSeance_day"].dt.date == jour_dt]
        titre = f"Personnel politique mobilisant le plus en % la 'République' le {jour_dt} (Seuil de {seuil_min_true} occurences)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["dateSeance_day"] >= debut) & (df["dateSeance_day"] <= fin)]
        titre = f"Personnel politique mobilisant le plus en % la 'République' du {debut.date()} au {fin.date()} (Seuil de {seuil_min_true} occurences)"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df["dateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Personnel politique mobilisant le plus la 'République' en % lors de X évènement (Seuil de {seuil_min_true} occurences)"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des occurrences ---
    counts = (
        df_filtered.groupby("nom_orateur_clean")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # --- Filtrage par seuil ---
    filtered = counts[counts["true_mentions"] >= seuil_min_true]

    # --- Trier par proportion décroissante et garder top_n ---
    df_top = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # --- Graphique ---
    fig = px.bar(
        df_top,
        x="nom_orateur_clean",
        y="proportion_true",
        hover_data=["true_mentions", "total_mentions"],
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False, yaxis_title="Proportion des interventions totales", xaxis_title="Personnel politique")
    fig.show()

    return df_top


In [37]:
# Top 10 orateurs en % par année
top_orateurs_periodique(df, periode="annee", annee=2024, seuil_min_true=20)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Gabriel Attal,90,20,0.222222
1,M. Gérald Darmanin,115,22,0.191304


In [38]:
# Top 10 orateurs sur la semaine X de X
top_orateurs_periodique(df, periode="semaine", annee=2021, semaine=6, seuil_min_true=10)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Jean-Michel Blanquer,58,20,0.344828
1,M. Gérald Darmanin,58,14,0.241379
2,M. Jean-Christophe Lagarde,47,10,0.212766
3,M. Alexis Corbière,101,15,0.148515
4,Mme Anne Brugnera,109,14,0.128440


In [39]:
# Top 10 orateurs en % (tel jour) le X 
top_orateurs_periodique(df, periode="jour", jour="2021-02-01", seuil_min_true=10)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true


In [40]:
# Top 10 orateurs en % (sur telle période) du 1er au 16 février 2021
top_orateurs_periodique(df, periode="intervalle", date_debut="2021-02-01", date_fin="2021-02-16", seuil_min_true=10)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Pierre-Yves Bournazel,35,16,0.457143
1,M. Guillaume Vuilletet,34,13,0.382353
2,M. Jean-Michel Blanquer,63,21,0.333333
3,M. Éric Poulliat,67,22,0.328358
4,M. Jean-Christophe Lagarde,87,24,0.275862
5,Mme Marlène Schiappa,129,33,0.255814
6,M. Gérald Darmanin,193,46,0.238342
7,M. Xavier Breton,72,17,0.236111
8,M. François de Rugy,91,19,0.208791
9,M. François Pupponi,67,13,0.194030


In [41]:
# Top 10 orateurs en % sur des jours spécifiques
top_orateurs_periodique(df, periode="jours", jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-4", "2021-02-05"], seuil_min_true=10)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,Mme Marlène Schiappa,56,28,0.500000
1,M. Éric Poulliat,44,16,0.363636
2,M. Jean-Christophe Lagarde,40,14,0.350000
3,M. Gérald Darmanin,131,30,0.229008
4,M. François de Rugy,60,13,0.216667
5,M. Boris Vallaud,61,13,0.213115
6,M. Florent Boudié,116,13,0.112069
7,M. Alexis Corbière,130,11,0.084615


#### Analyse discussion du projet de loi séparatisme

In [42]:
# Top 10 orateurs lors de la 1ère discussion du projet de loi séparatisme 
top_orateurs_periodique(df, periode="jours", jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16"], seuil_min_true=10)

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Pierre-Yves Bournazel,33,16,0.484848
1,M. Guillaume Vuilletet,34,13,0.382353
2,M. Jean-Michel Blanquer,60,20,0.333333
3,M. Éric Poulliat,67,22,0.328358
4,M. Jean-Christophe Lagarde,75,24,0.320000
5,Mme Marlène Schiappa,114,33,0.289474
6,M. Gérald Darmanin,191,46,0.240838
7,M. Xavier Breton,72,17,0.236111
8,M. François de Rugy,91,19,0.208791
9,M. François Pupponi,67,13,0.194030


In [43]:
# Top 10 orateurs en % lors des discussions du projet de loi séparatisme 
top_orateurs_periodique(df, periode="jours", jours=["2021-02-01", "2021-02-02", "2021-02-03", "2021-02-04", "2021-02-05", "2021-02-08", "2021-02-10", "2021-02-11", "2021-02-12", "2021-02-13", "2021-02-16", "2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], seuil_min_true=10)



,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Pierre-Yves Bournazel,38,19,0.500000
1,M. Guillaume Vuilletet,50,18,0.360000
2,M. Éric Poulliat,85,30,0.352941
3,M. Jean-Michel Blanquer,66,22,0.333333
4,M. Jean-Christophe Lagarde,75,24,0.320000
5,M. Alain Bruneel,41,13,0.317073
6,M. Francis Chouat,46,12,0.260870
7,Mme Marlène Schiappa,147,37,0.251701
8,M. Sacha Houlié,41,10,0.243902
9,M. Gérald Darmanin,194,47,0.242268


In [44]:
# Top 10 orateurs en % lors de la deuxième phase de discussion post débat Sénat 
top_orateurs_periodique(df, periode="jours", jours=["2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], seuil_min_true=10)



,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,M. Julien Ravier,37,12,0.324324
1,M. Alexis Corbière,65,14,0.215385


### Évolution dans le temps de la fréquence d'utilisation de personnes spécifiques

In [45]:
def stats_orateur_proportion(df, orateur, periode="semaine", colonne_condition="repu_match_valide"):
  
    # Filtrer sur l'orateur choisi
    df_orateur = df[df["nom_orateur_clean"] == orateur].copy()
    if df_orateur.empty:
        raise ValueError(f"Aucune donnée trouvée pour l'orateur : {orateur}")

    # Définir la granularité
    if periode == "semaine":
        df_orateur["periode"] = (
            df_orateur["dateSeance_day"].dt.isocalendar().year.astype(str)
            + "-W"
            + df_orateur["dateSeance_day"].dt.isocalendar().week.astype(str)
        )
    elif periode == "mois":
        df_orateur["periode"] = df_orateur["dateSeance_day"].dt.to_period("M").astype(str)
    elif periode == "annee":
        df_orateur["periode"] = df_orateur["dateSeance_day"].dt.year.astype(str)
    else:
        raise ValueError("periode doit être 'semaine', 'mois' ou 'annee'")

    # Compter occurrences par période (total et True)
    df_counts = (
        df_orateur.groupby("periode")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Proportion de True
    df_counts["proportion_true"] = df_counts["true_mentions"] / df_counts["total_mentions"]

    # Moyenne et médiane des proportions
    moyenne_prop = df_counts["proportion_true"].mean()
    mediane_prop = df_counts["proportion_true"].median()

    return {
        "moyenne_proportion": moyenne_prop,
        "mediane_proportion": mediane_prop,
        "df_counts": df_counts,
    }


In [46]:
# Définir la personne recherchée 
orateur = "M. Gérald Darmanin"

# Statistiques hebdomadaires sur "République"
stats = stats_orateur_proportion(df, orateur, periode="annee", colonne_condition="repu_match_valide")

print(f"{orateur} - Moyenne/semaine (proportion)  : {stats['moyenne_proportion']:.2f}")
print(f"{orateur} - Médiane/semaine (proportion)  : {stats['mediane_proportion']:.2f}")

# Voir le détail par période
print(stats["df_counts"].head())

M. Gérald Darmanin - Moyenne/semaine (proportion)  : 0.13
M. Gérald Darmanin - Médiane/semaine (proportion)  : 0.14
  periode  total_mentions  true_mentions  proportion_true
0    2017             644             19         0.029503
1    2018             810             32         0.039506
2    2019             368             20         0.054348
3    2020             295             54         0.183051
4    2021             346             84         0.242775


In [47]:
# Définir la personne recherchée 
orateur = "M. Pierre-Yves Bournazel"
colonne_condition = "repu_match_valide"  # colonne booléenne
periode = "M" 

# Filtrer le DataFrame sur la personne choisie
df_orateur = df[df["nom_orateur_clean"] == orateur].copy()

# S'assurer que la colonne date est bien en datetime
df_orateur["dateSeance_day"] = pd.to_datetime(df_orateur["dateSeance_day"])

# Calculer les occurrences mensuelles
df_monthly = (
    df_orateur
    .resample(periode, on="dateSeance_day")[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

# Calculer la proportion de True
df_monthly["proportion_true"] = df_monthly["true_mentions"] / df_monthly["total_mentions"]

# Tracer le graphique en proportion
fig_time = px.bar(
    df_monthly,
    x="dateSeance_day",
    y="proportion_true",
    title=f"Évolution mensuelle de la proportion des utilisations de la 'République' par {orateur}",
    labels={"dateSeance_day": "Mois", "proportion_true": "Proportion de mentions valides"},
    template="plotly_white",
)

fig_time.update_layout(
    yaxis_tickformat=".0%",  # affichage en pourcentage
    showlegend=False
)

fig_time.show()


/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3294/2629417355.py:15: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [48]:
# Définir les paramètres
orateur = "M. Pierre Meurin"
colonne_condition = "repu_match_valide"
periode = "M"  # "M" = mois, "W" = semaine, etc.

# Filtrer les données pour l'orateur
df_orateur = df[df["nom_orateur_clean"] == orateur].copy()

# S'assurer que la colonne date est bien en datetime
df_orateur["dateSeance_day"] = pd.to_datetime(df_orateur["dateSeance_day"])

# Agréger les données par mois
df_monthly = (
    df_orateur
    .resample(periode, on="dateSeance_day")[colonne_condition]
    .agg(
        total_mentions="count",
        true_mentions=lambda x: (x == True).sum()
    )
    .reset_index()
)

# Calculer la proportion
df_monthly["proportion_true"] = df_monthly["true_mentions"] / df_monthly["total_mentions"]

# Créer la figure avec Plotly Graph Objects
fig = go.Figure()

# Barres : volume total
fig.add_trace(go.Bar(
    x=df_monthly["dateSeance_day"],
    y=df_monthly["true_mentions"],
    name="Nombre total des occurrences",
    marker_color="rgba(99, 110, 250, 0.6)",
    yaxis="y1"
))

# Courbe : proportion
fig.add_trace(go.Scatter(
    x=df_monthly["dateSeance_day"],
    y=df_monthly["proportion_true"] * 100,  # en pourcentage
    name="Proportions des occurrences",
    mode="lines+markers",
    line=dict(color="rgba(239, 85, 59, 0.8)", width=3),
    yaxis="y2"
))

# Mise en forme du graphique
fig.update_layout(
    title=f"Évolution mensuelle des occurrences de la 'République' par {orateur}",
    xaxis=dict(title="Mois"),
    yaxis=dict(
        title="Nombre total des occurrences",
        showgrid=False
    ),
    yaxis2=dict(
        title="Proportion des occurences sur total",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.7)"),
    template="plotly_white",
    bargap=0.2
)

fig.show()


/var/folders/gc/0c_w194n61ggswq_j8v40mj80000gn/T/ipykernel_3294/472484391.py:15: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



## Personnel politique par groupes

In [49]:
# Fonction générale 

# Définir la personne recherchée 
parti = "ECO"

# Filtrer sur le parti choisi
df_partis = df[df["groupe_députés_affiliation"] == parti]

# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df_partis.groupby("nom_orateur_clean")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 15]

# Trier par proportion décroissante et garder les  premiers
df_top10_partis = filtered.sort_values("proportion_true", ascending=False).head(20).reset_index(drop=True)

# Figure
fig_top_orateurs_partis = px.bar(
   df_top10_partis,
    x="nom_orateur_clean",
    y="proportion_true",
    title=f"Top 10 des député.es {parti} en proportions des mentions de la 'République' (15 occurrences min)",
    labels={"nom_orateur_clean": "Député.es", "proportion_true": "% 'République'"},
    template="plotly_white",
)
fig_top_orateurs_partis.update_layout(
    xaxis_tickangle=-45,
    showlegend=False,
)
fig_top_orateurs_partis.show()

df_top10_partis

,nom_orateur_clean,total_mentions,true_mentions,proportion_true
0,Mme Cyrielle Chatelain,364,19,0.052198
1,M. Benjamin Lucas,2481,104,0.041919
2,Mme Sophie Taillé-Polian,428,17,0.039720
3,Mme Sandra Regol,881,21,0.023837


In [50]:
# TODO : changer manuellement les couleurs des graphiques

def evolution_députés_partis_pourcentage(df,
                                  date_col="dateSeance_day",
                                  personnel_col="nom_orateur_clean",
                                  match_col="repu_match_valide",
                                  min_true_mentions=150,
                                  top_n=6):

    # Calcul global des proportions par groupe
    counts = (
        df.groupby(personnel_col)[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion globale
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # Filtrer les partis avec au moins min_true_mentions
    filtered = counts[counts["true_mentions"] >= min_true_mentions]

    # Sélectionner les top_n groupes selon la proportion
    top_personnel = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)[personnel_col]
        .tolist()
    )

    # Filtrer les données pour ces groupes
    df_top = df[df[personnel_col].isin(top_personnel)].copy()

    # Calculer les stats annuelles : mentions totales et vraies
    df_grouped = (
        df_top.groupby([pd.Grouper(key=date_col, freq="Y"), personnel_col])[match_col]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )

    # Ajouter la proportion annuelle (% d'utilisation)
    df_grouped["proportion_true"] = (
        df_grouped["true_mentions"] / df_grouped["total_mentions"]
    )

    # Extraire l'année pour affichage
    df_grouped["Année"] = df_grouped[date_col].dt.year

    # Tracer l'évolution du % dans le temps
    fig = px.line(
        df_grouped,
        x="Année",
        y="proportion_true",
        color=personnel_col,
        markers=True,
        title=f"Évolution annuelle du % d'utilisation de la 'République' des {top_n} principaux personnels politiques (>{min_true_mentions} occurrences)",
        labels={
            "proportion_true": "% d'utilisation",
            personnel_col: "Groupe parlementaire"
        }
    )

    fig.update_layout(
        xaxis=dict(dtick=1),
        hovermode="x unified",
        template="plotly_white",
        legend_title_text="Groupe",
        yaxis_tickformat=".0%",
    )

    fig.show()


#### tentative en cours d'avoir des stats plus fines par partis.

In [51]:
# Définir la personne recherchée 
parti = "LFI"

# Filtrer sur le parti choisi
df_partis = df[df["parti_affiliation"] == parti]

def top_orateurs_partis_periodique(df_partis, periode="annee", annee=None, semaine=None, jour=None, 
                            date_debut=None, date_fin=None, jours=None,
                            colonne_condition="repu_match_valide",
                            seuil_min_true=None, top_n=20):

    # --- Filtrage selon la période ---
    if periode == "annee":
        if annee is None:
            raise ValueError("Il faut préciser l'année pour periode='annee'")
        df_filtered = df[df["dateSeance_day"].dt.year == annee]
        titre = f"Personnel politique mobilisant en % le plus la 'République' en {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true} occurences)"

    elif periode == "semaine":
        if annee is None or semaine is None:
            raise ValueError("Il faut préciser l'année et la semaine pour periode='semaine'")
        df_filtered = df[
            (df["dateSeance_day"].dt.isocalendar().year == annee) &
            (df["dateSeance_day"].dt.isocalendar().week == semaine)
        ]
        titre = f"Personnel politique mobilisant en % le plus la 'République' la {semaine} semaine {annee} à l'Assemblée Nationale (Seuil de {seuil_min_true})"

    elif periode == "jour":
        if jour is None:
            raise ValueError("Il faut préciser la date pour periode='jour'")
        jour_dt = pd.to_datetime(jour).date()
        df_filtered = df[df["dateSeance_day"].dt.date == jour_dt]
        titre = f"Personnel politique mobilisant le plus en % la 'République' le {jour_dt} (Seuil de {seuil_min_true} occurences)"

    elif periode == "intervalle":
        if date_debut is None or date_fin is None:
            raise ValueError("Il faut préciser date_debut et date_fin pour periode='intervalle'")
        debut = pd.to_datetime(date_debut)
        fin = pd.to_datetime(date_fin)
        df_filtered = df[(df["dateSeance_day"] >= debut) & (df["dateSeance_day"] <= fin)]
        titre = f"Personnel politique mobilisant le plus en % la 'République' du {debut.date()} au {fin.date()} (Seuil de {seuil_min_true} occurences)"

    elif periode == "jours":
        if jours is None or not isinstance(jours, (list, tuple)):
            raise ValueError("Il faut fournir une liste de dates pour periode='jours'")
        jours_dt = [pd.to_datetime(j).date() for j in jours]
        df_filtered = df[df["dateSeance_day"].dt.date.isin(jours_dt)]
        titre = f"Personnel politique mobilisant le plus la 'République' en % lors de X évènement (Seuil de {seuil_min_true} occurences)"

    else:
        raise ValueError("periode doit être 'annee', 'semaine', 'jour', 'intervalle' ou 'jours'")

    # --- Comptage des occurrences ---
    counts = (
        df_filtered.groupby("nom_orateur_clean")[colonne_condition]
        .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
        .reset_index()
    )
    counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

    # --- Filtrage par seuil ---
    filtered = counts[counts["true_mentions"] >= seuil_min_true]

    # --- Trier par proportion décroissante et garder top_n ---
    df_top = (
        filtered.sort_values("proportion_true", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    # --- Graphique ---
    fig = px.bar(
        df_top,
        x="nom_orateur_clean",
        y="proportion_true",
        hover_data=["true_mentions", "total_mentions"],
        title=titre,
        template="plotly_white",
    )
    fig.update_layout(xaxis_tickangle=-45, showlegend=False, yaxis_title="Proportion des interventions totales", xaxis_title="Personnel politique")
    fig.show()

    return df_top


KeyError: 'parti_affiliation'

In [ ]:
# Top 10 orateurs en % lors de la deuxième phase de discussion post débat Sénat 
top_orateurs_partis_periodique(df_partis, periode="jours", jours=["2021-06-28", "2021-06-29", "2021-06-30", "2021-07-01", "2021-01-23"], seuil_min_true=5)

## Autres variables

--> en réalité ce qui serait utile ce serait de faire des vraies stats/régressions pour mesurer le poids de chacune de ces variables sur la probabilité d'utiliser la république. À voir comment faire 

In [ ]:
df["civ"] = df["civ"].replace({"M.": "Homme", "Mme": "Femme"})

In [ ]:
df.to_csv(
    "../data/interim/df_identification_republi_simple.csv",
    index=False,
)

In [ ]:
fig = px.bar(df["civ"].value_counts())
fig.update_layout(
    title="Répartition des genres (civ)", template="plotly_white", showlegend=False
)
fig.show()

### Autres variables 

In [ ]:
# Nécessité ici de transformer l'âge en chiffre pour l'ordonner. 

# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df.groupby("experienceDepute")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_experience_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

df_experience_top20

# Graphique
fig_experience = px.bar(
    df_experience_top20,
    x="experienceDepute",
    y="proportion_true",
    title="Évolution de la proportion d'utilisation de la 'République avec l'expérience",
    labels={"experienceDepute": "Expérience", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_experience.update_layout(xaxis_tickangle=-45)

fig_experience.show()

df_experience_top20

In [ ]:
# Compter le nombre de fois où chaque orateur dit "République"
counts = (
    df.groupby("age")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_experienceb_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

# Graphique
fig_experienceb = px.bar(
    df_experienceb_top20,
    x="age",
    y="proportion_true",
    title="Évolution de la proportion d'utilisation de la 'République avec l'âge",
    labels={"age": "Age", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_experienceb.update_layout(xaxis_tickangle=-45)

fig_experienceb.show()

df_experienceb_top20

In [ ]:
# Compter le nombre de fois où chaque orateur d'un département dit "République"
counts = (
    df.groupby("departementCode")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_dpt_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

df_dpt_top20

# Graphique
fig_dpt = px.bar(
    df_dpt_top20,
    x="departementCode",
    y="proportion_true",
    title="Évolution de la proportion d'utilisation de la 'République par département",
    labels={"departementCode": "Departement", "proportion_true": "% 'République'"},
    template="plotly_white",
)

fig_dpt.update_layout(xaxis_tickangle=-45)

fig_dpt.show()

df_dpt_top20

==> en VA ce sont les départements IDF et surtout le 93 de LFI qui reviennent le plus, mais en % ce sont les territoires d'outre-mers

In [ ]:
# Compter le nombre de fois où chaque orateur/genre dit "République"
counts = (
    df.groupby("civ")["repu_match_valide"]
    .agg(total_mentions="count", true_mentions=lambda x: (x == True).sum())
    .reset_index()
)

# Calcul de la proportion
counts["proportion_true"] = counts["true_mentions"] / counts["total_mentions"]

# Filtrer les orateurs avec au moins 10 mentions True
filtered = counts[counts["true_mentions"] >= 50]

# Trier par proportion décroissante et garder les 40 premiers
df_genre_top20 = filtered.sort_values("proportion_true", ascending=False).head(40).reset_index(drop=True)

df_genre_top20

## Test de régressions linéaires 

### Étape 1 : restructuration des variables 

In [ ]:
# Réfléchir à recoder l'âge en génération (reprendre celles de VT ?)

In [ ]:
df["genre"] = df["civ"].replace({"Homme": "1", "Femme": "0"})

In [ ]:
df["repu"] = df["repu_match_valide"].replace({"True": "1", "False": "0"})

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

def regression_lineaire(df, y_col, x_cols):
  
    # Définir X et y
    X = df[x_cols]
    y = df[y_col]
    
    # Encoder les colonnes catégorielles si nécessaire
    X = pd.get_dummies(X, drop_first=True)
    
    # Forcer en 2D si une seule variable explicative
    if X.shape[1] == 1:
        X = X.values.reshape(-1, 1)
    else:
        X = X.values
    
    # y doit être 1D
    y = y.values
    
    # Créer et entraîner le modèle
    model = LinearRegression()
    model.fit(X, y)
    
    # Prédictions
    y_pred = model.predict(X)
    
    # Résultats
    print("Coefficient(s):", model.coef_)
    print("Intercept:", model.intercept_)
    print("Score R²:", model.score(X, y))
    
    # Graphique si une seule variable explicative
    if X.shape[1] == 1:
        plt.scatter(X, y, color="blue", label="Données réelles")
        plt.plot(X, y_pred, color="red", label="Régression")
        plt.xlabel(x_cols[0])
        plt.ylabel(y_col)
        plt.legend()
        plt.show()
    
    return model


In [ ]:
modele = regression_lineaire(df, y_col="repu_match_valide", x_cols=["parti_affiliation"])

In [ ]:
modele = regression_lineaire(df, y_col="repu_match_valide", x_cols=["experienceDepute"])

In [ ]:
modele = regression_lineaire(df, y_col="repu_match_valide", x_cols=["parti_affiliation", "civ", "experienceDepute"])

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

def regression_lineaire_df(df, y_col, x_cols, summary=False):
 
    # Définir X (variable à expliquer) et y (variable explicatives)
    X = df[x_cols]
    y = df[y_col]
    
    # Encoder les colonnes catégorielles si nécessaire
    X = pd.get_dummies(X, drop_first=True)
    
    # Forcer en numpy array
    X_values = X.values
    y_values = y.values
    
    # ----- Version scikit-learn -----
    model = LinearRegression()
    model.fit(X_values, y_values)
    y_pred = model.predict(X_values)
    
    print("Régression (scikit-learn)")
    print("Coefficient(s):", model.coef_)
    print("Intercept:", model.intercept_)
    print("Score R²:", model.score(X_values, y_values))
    
    # Graphique si une seule variable explicative
    if X.shape[1] == 1:
        plt.scatter(X_values, y_values, color="blue", label="Données réelles")
        plt.plot(X_values, y_pred, color="red", label="Régression")
        plt.xlabel(x_cols[0])
        plt.ylabel(y_col)
        plt.legend()
        plt.show()
    
    # ----- Version statsmodels -----
    if summary:
        X_sm = sm.add_constant(X)  # ajoute la constante pour l'intercept
        model_sm = sm.OLS(y, X_sm).fit()
        print("Résumé statistique (statsmodels):")
        print(model_sm.summary())
        return model, model_sm
    
    return model


In [ ]:
modele_sklearn, modele_stats = regression_lineaire_df(df, y_col="repu_match_valide", x_cols=["parti_affiliation", "civ"], summary=True)